**Avvio locale e Colab:** preparare la cartella completa seguendo il [README del laboratorio](../README.md). Conservare gli identificativi delle celle e salvare gli output nel proprio runtime.

<!-- C01 -->
# Detection Studio
Computer Vision · Lezione 16

### Elaborato individuale — da compilare


Il notebook della lezione 15 è una dimostrazione completa da consultare. Qui sono richieste **cinque attività guidate**: i tre TODO di codice **C07, C12, C19** e i richiami scritti **P01 in C18** (IoU e matching), **P02 in C26** (punto operativo e AP). I due richiami non richiedono nuovo codice, training, inferenze o download. Conservare i **40 ID C01–C40** e le **23 celle di codice**.

Il solo training richiesto resta quello già fissato di tre epoche; il confronto iniziale/finale e le inferenze del percorso sono già predisposti. CPU resta disponibile e CUDA è opzionale se presente; una corsa su Colab ha il proprio ambiente e non eredita i tempi CPU locali del riferimento.

Rispondere a **N01–N08 in 3–5 righe o una piccola tabella**, richiamando gli output senza ricopiarli. N05/N06 possono citare P02 per il ruolo della soglia, senza rifare il calcolo. N06 approfondisce **una sola scena** tra le tre già prodotte da C31. Scrivere **F01–F05 direttamente in C36, entro 500 parole complessive**: nessun PDF o report separato. F04 richiama le immagini C31 e la risposta N06, senza nuove figure né tre analisi scritte aggiuntive. D01–D08, discussione conclusiva, repliche e challenge ulteriori sono **facoltativi e non valutati**.

Consegnare il `.ipynb` con identità, completamenti, risposte e output insieme alla cartella `output_lezione16/` compressa da C39. Nome suggerito: `ComputerVision_Lezione16.ipynb`. Se si usano risultati di recupero, dichiarare quali operazioni sono personali e quali misure sono fornite nell’archivio originale; non presentarle come nuovo training o come misure GPU/cloud.

Adattamento reale della sola testa finale di un detector COCO. **Cxx conta tutte le celle**, anche il testo. La corsa di riferimento è CPU. Su Colab il notebook può usare una GPU CUDA disponibile e registra il dispositivo. Le metriche e i tempi possono differire.

Nei materiali del corso si usa il **punto decimale** (0.5) e la **virgola seguita da uno spazio** fra componenti di vettori e argomenti, per esempio (1.2, 1, −0.3). I conteggi interi non usano separatori delle migliaia.

<!-- C02 -->
## Protocollo fissato

**Lessico prima dell’avvio.** Holdout è l’insieme escluso dal fit e dalle scelte del protocollo. Pretrained significa pesi appresi su dati sorgente; predictor è l’ultimo classificatore e regressore delle regioni. GT sono le annotazioni di riferimento, AP è Average Precision e supporto il numero di GT valide. FPN, RPN, RoI e NMS riprendono la lezione 15.

Penn 96 immagini training, Fudan 74 holdout per sorgente di acquisizione. Tre epoche prefissate, 48 aggiornamenti/epoca, nessuna scelta su holdout. Nuova testa a due classi (background/person), restante detector congelato. Confronto epoch 0 ed epoch 3 della **stessa inizializzazione**. Una sola classe di oggetti, nessuna misura multicategoria. Le maschere della release contengono 423 istanze (263+160), mentre la pagina storica riporta 345.

In [ ]:
# C03
from pathlib import Path
import sys, os, json, copy, time, zipfile, importlib.util, subprocess, io
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import files
    root=Path('/content/ComputerVision_Lezione16')
    candidates=list(root.glob('**/supporto/detection_utils.py')) if root.exists() else []
    if not candidates:
        print('Carica Pacchetto_studenti_Lezione_16.zip. Il solo file ipynb non contiene dati o supporto.')
        upload=files.upload()
        zips=[name for name in upload if name.endswith('.zip')]
        if len(zips)!=1: raise RuntimeError('Caricare un solo ZIP studenti della lezione 16.')
        root.mkdir(exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(upload[zips[0]])) as z:
            for name in z.namelist():
                if not (root/name).resolve().is_relative_to(root.resolve()): raise ValueError('Percorso ZIP non sicuro')
            z.extractall(root)
        candidates=list(root.glob('**/supporto/detection_utils.py'))
    if len(candidates)!=1: raise RuntimeError('Pacchetto non riconosciuto: serve una sola cartella supporto.')
    BASE=candidates[0].parent.parent
    os.chdir(BASE/'studenti')
    if importlib.util.find_spec('pycocotools') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','--no-deps','pycocotools==2.0.11'])
else:
    BASE=Path.cwd().parent
    if not (BASE/'supporto/detection_utils.py').exists(): raise RuntimeError('Aprire dalla cartella studenti o docente del pacchetto estratto.')
import numpy as np
import torch, torchvision
from PIL import Image
from torchvision.ops import nms
assert nms(torch.tensor([[0.,0.,2.,2.]]),torch.tensor([.9]),.5).tolist()==[0]
sys.path.insert(0,str(BASE/'supporto'))
import detection_utils as u
DEVICE=os.environ.get('CV_DEVICE','auto')  # auto sceglie CUDA disponibile, altrimenti CPU
ENV=u.setup(DEVICE);DEVICE=ENV['device']
OUT=Path.cwd()/'output_lezione16';OUT.mkdir(exist_ok=True)
print(ENV)
print('Protocollo:',u.CONFIG)


<!-- C04 -->
## Preparazione e diritti

**Impronte e confronto.** Hash indica un’impronta compatta del contenuto. dHash è un’impronta percettiva binaria dei confronti fra intensità vicine; la distanza di Hamming conta i bit diversi. Questo screening è distinto dal confronto di pixel identici e non certifica indipendenza delle scene.

C05 scarica circa 51 MiB di PennFudan e 74 MiB di pesi ufficiali la prima volta. Riusa la cache nelle corse successive. Il dataset conserva i diritti degli autori: la fonte non concede una licenza aperta di redistribuzione. Lo ZIP studenti contiene il downloader e non il dataset o i pesi COCO. Preparare rete/cache **prima** dell’esecuzione. Su Colab una nuova VM può richiedere di nuovo i download. La copia locale dell’archivio originale permette il laboratorio senza rete.

Nella propria copia completare C07/C12/C19 individualmente in autonomia prima di eseguire tutto in ordine. Non avviare un Run All con TODO incompleti.

In [ ]:
# C05
PROVENANCE=u.prepare(BASE,download=True)
manifest=json.loads((BASE/"manifest.json").read_text())
assert PROVENANCE == manifest["provenance"], "Dataset o pesi diversi dal riferimento: verificare la cache e il manifest."
print(json.dumps(PROVENANCE,indent=2))
print("Hash di dataset e pesi verificati contro il manifest.")


<!-- C06 -->
## N01 · Box dalla maschera

**Tipi e assi.** Tensore significa array multidimensionale. `float32` rappresenta reali a 32 bit, `int64` interi a 64 bit. `[N, 4]` significa N istanze con quattro bordi ciascuna; `[3, H, W]` significa canali RGB, altezza e larghezza. Gli ID della maschera sono locali all’immagine, non identità personali.

**Completa C07.** Per ogni IDpositivo della maschera produci il rettangolo minimo in coordinate semichiuse: minima coordinata inclusa, massima+1. Lo sfondo 0 non è un oggetto. Il risultato deve avere shape[N, 4] e dtypefloat32. Per pixel x=10…19, y=20…39, quali sono box e area?

**Risposta N01 (3–5 righe o piccola tabella):** [scrivere qui; citare gli output senza ricopiarli]

In [ ]:
# C07 · TODO
# Definire mask_to_boxes(mask), output float32[N,4], semichiuso, e assegnare u.mask_to_boxes.
raise NotImplementedError("Completare C07")

In [ ]:
# C08
train=u.Pedestrians(BASE,'Penn');holdout=u.Pedestrians(BASE,'Fudan')
AUDIT=u.split_audit(train,holdout)
print('Train:',{k:v for k,v in train.summary().items() if k!='files'})
print('Holdout:',{k:v for k,v in holdout.summary().items() if k!='files'})
print('Audit:',AUDIT)
assert len(train)==96 and len(holdout)==74
assert train.summary()['instances']==263 and holdout.summary()['instances']==160

In [ ]:
# C09
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,ds in zip(axes,[train,holdout]):
    im,t=ds[0];ax.imshow(im.permute(1,2,0))
    for x1,y1,x2,y2 in t['boxes'].tolist():ax.add_patch(Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor='#00bdc7',linewidth=2))
    ax.set_title(ds.files[0].name);ax.axis('off')
fig.tight_layout();fig.savefig(OUT/'dataset_examples.png',dpi=160);plt.show()

<!-- C10 -->
## N02 · Split e supporto

**Richiamo.** Supporto è il numero di GT valide di un gruppo. Dominio indica le condizioni di acquisizione, non il nome della categoria. Distanza dHash 11 su 64 significa 11 bit diversi nella coppia minima osservata, non 11 pixel né una soglia universale di duplicazione.

Perché il box non è l’unità da separare? Quale rischio riduce il raggruppamento Penn/Fudan e quale differenza di dominio introduce? L’audit dei pixel identici e dHash è uno screening, non prova generale di assenza di immagini simili. Nel train non ci sono target small, nel holdout sono 5: come limita l’interpretazione di APsmall?

**Risposta N02 (3–5 righe o piccola tabella):** [scrivere qui; citare gli output senza ricopiarli]

<!-- C11 -->
## Predictor adattata

**Ruoli distinti.** La box head costruisce 1024 feature per RoI; il predictor finale converte quel vettore in score di classe e correzioni dei box. Una RoI è una Region of Interest, regione candidata. Logits sono score prima della softmax; gli otto valori geometrici sono due gruppi di quattro offset, uno per classe interna.

Caricamento COCO_V1 completo, quindi sostituzione della sola `roi_heads.box_predictor`. Due classi producono 2 logits e 8 coordinate di regressione. Feature 1024: (1024+1)×10=10250 parametri. `FrozenBatchNorm 2 d` conserva statistiche e affine pretrained. Backbone, FPN, RPN, ROI box_head rimangono congelati. **Non è fine-tuning dell’intera rete.**

In [ ]:
# C12 · TODO
# Definire configure_trainable(model), congelare tutto, attivare solo roi_heads.box_predictor e assegnare u.configure_trainable.
raise NotImplementedError("Completare C12")

In [ ]:
# C13
model=u.make_model(BASE,DEVICE)
trainable=configure_trainable(model)
print('Parametri trainabili:',trainable)
TOTAL=sum(p.numel() for p in model.parameters());TRAINABLE=sum(p.numel() for p in model.parameters() if p.requires_grad)
assert TRAINABLE==10250
print('Totali:',TOTAL,'trainabili:',TRAINABLE)
torch.save({k:v.cpu() for k,v in model.roi_heads.box_predictor.state_dict().items()},OUT/'predictor_epoch0.pt')

<!-- C14 -->
## N03 · Congelamento e modalità

**Stato e modalità.** Parametri sono coefficienti appresi, buffer altro stato conservato dal modello. Un invariante è una quantità che deve rimanere uguale prima e dopo il fit. Congelare i gradienti non mette automaticamente un modulo in eval.

`requires_grad=False` controlla i gradienti dei parametri. `train()/eval()` controlla il comportamento dei moduli. Per ottenere le loss il detector è in train, ma RPN e feature extractor sono in eval. La RPN produce le stesse 150 proposte massime di inferenza. La ROI head in train aggiunge GT e campiona proposte per supervisionare la testa di predizione. Queste operazioni non avvengono in eval, dove il modello non riceve target.

**Risposta N03 (3–5 righe o piccola tabella):** [scrivere qui; citare gli output senza ricopiarli]

In [ ]:
# C15
image,target=train[0]
print('Input originale:',tuple(image.shape),'box:',target['boxes'].shape)
model.eval()
transformed,targets_transformed=model.transform([image.to(DEVICE)],[{k:v.to(DEVICE) for k,v in target.items()}])
print('Dimensione dopo resize:',transformed.image_sizes,'tensore con padding:',tuple(transformed.tensors.shape))
print('Primo box originale:',target['boxes'][0].tolist())
print('Primo box ridimensionato:',targets_transformed[0]['boxes'][0].tolist())

<!-- C16 -->
## Baseline della stessa testa nuova
C17 valuta la testa appena inizializzata sul holdout. Il backbone sa già estrarre feature COCO, ma la nuova testa di predizione non ha ancora appreso persona/background. Il confronto epoch 0/epoch 3 misura l’adattamento della testa in questo protocollo. Non misura il guadagno sul detector COCO originale.

In [ ]:
# C17
pred0,time0=u.predict(model,holdout,DEVICE)
print('Immagini valutate:',len(pred0),'tempo inferenza [s]:',time0)
print('Output:',{k:tuple(v.shape) for k,v in pred0[0].items()})

<!-- C18 -->
## N04 · Esportazione COCO
**Completa C19.** Converti coordinate xyxy semichiuse in xywh senza sottrarre 1. Il box[10, 20, 20, 40] diventa[10, 20, 10, 20]. I target mantengono area del box originale. `category_id=1`, `iscrowd=0`. Per AP conserviamo tutte le predizioni prodotte sopra la soglia interna.001: non applichiamo la soglia di visualizzazione.5.

### P01 · IoU e matching uno-a-uno

Richiamo della lezione 15, con box continui/semichiusi `xyxy`, una sola classe e una sola immagine. Il GT è G=(0, 0, 10, 10). Le due predizioni sono A=G con score 0.9 e B=(1, 0, 11, 10) con score 0.8. Nel toy **non si applica NMS**, né crowd/ignore.

Calcolare area dei box, intersezione, unione e IoU fra A e B (A coincide con G). Applicare poi il matching valutativo a IoU≥0.5, in ordine score decrescente, con ogni GT abbinabile una sola volta: quante TP e FP si ottengono? Perché due box che superano la soglia non significano due TP sullo stesso oggetto? Non introdurre il `+1` nelle aree continue.

**Risposta P01 (3–5 righe o piccola tabella):** [scrivere qui]


**Risposta N04 (3–5 righe o piccola tabella):** [scrivere qui; citare gli output senza ricopiarli]

In [ ]:
# C19 · TODO
# Definire xyxy_to_xywh(box), differenze senza -1, e assegnare u.xyxy_to_xywh.
raise NotImplementedError("Completare C19")

In [ ]:
# C20
metrics0=u.coco_eval(holdout,pred0)
print({k:metrics0[k] for k in ['AP','AP50','AP75','AP_small','AP_medium','AP_large','AR100']})
print(metrics0['evaluation'])

In [ ]:
# C21
assert metrics0['evaluation']['recThrs_n']==101
assert len(metrics0['evaluation']['iouThrs'])==10
assert metrics0['AP']==metrics0['AP_person']
print('Una categoria valutata: AP_person coincide con AP. Il background non entra nella media.')

<!-- C22 -->
## Training prefissato

**Richiamo prima del ciclo.** Un’epoca attraversa tutte le immagini Penn, uno step aggiorna i pesi una volta. Learning rate scala il passo SGD; momentum conserva memoria dei gradienti. Weight decay in questa SGD aggiunge al gradiente un contributo proporzionale ai pesi. Le loss sono classificazione delle RoI e Smooth L1 sugli errori degli offset positivi; `beta=1/9` è la soglia fra rami quadratico e lineare.

C23 addestra 3 epoche con SGD .005, momentum.9, weight_decay.0005, batch 2. Nessuna augmentation, 48 step/epoca. Due loss: classificazione ROI e regressione ROI. RPNcongelata in eval non calcola le sue loss. La verifica finale confronta tutti i parametri congelati e i buffer prima/dopo. Il checkpoint restituito è sempre epoch 3, senza selezione sul holdout.

In [ ]:
# C23
TRAINING=u.train_fixed(model,train,DEVICE,OUT)
print(TRAINING)
assert TRAINING['updates']==144 and TRAINING['frozen_parameters_unchanged'] and TRAINING['buffers_unchanged']

In [ ]:
# C24
pred3,time3=u.predict(model,holdout,DEVICE)
metrics3=u.coco_eval(holdout,pred3)
print('Inference holdout [s]:',time3)
for metric in ['AP','AP50','AP75','AP_small','AP_medium','AP_large','AR100']:
    a,b=metrics0[metric],metrics3[metric]
    print(metric,'iniziale',a,'epoch3',b,'delta pp',None if a is None or b is None else 100*(b-a))

In [ ]:
# C25
fig,ax=plt.subplots(figsize=(9,4))
for key in ['loss_classifier','loss_box_reg']:
    ax.plot([r['epoch'] for r in TRAINING['history']],[r[key] for r in TRAINING['history']],marker='o',label=key)
ax.set(xlabel='Epoca',ylabel='Loss media dei minibatch',xticks=[1,2,3]);ax.legend();ax.grid(alpha=.25);fig.tight_layout();fig.savefig(OUT/'training_losses.png',dpi=170);plt.show()

<!-- C26 -->
## N05 · AP e dimensioni

**Richiamo del valutatore.** Precision interpolata è l’inviluppo: per ogni recall prende la massima precisione fra i punti che raggiungono almeno quel recall. AP integra o campiona questo andamento secondo il protocollo dichiarato; AP50 fissa IoU a 0.50. N/A indica una metrica non definita per supporto assente.

I valori sono proporzioni 0…1. AP media 10 soglie IoU e 101 livelli di recall. AP50 e AP75 fissano la soglia IoU. small≤32², medium 32²…96², large≥96² in pixel quadrati originali, con convenzioni esatte dell’evaluatore ai bordi. Non filtriamo soltanto i GT: COCO gestisce ignore anche per matching e detections fuori fascia. **None/N/A significa supporto assente, non APzero.**

### P02 · Punto operativo e ranking AP

Caso sintetico, distinto da PennFudan: **una classe, due GT**, nessun crowd/ignore e matching uno-a-uno. Il ranking completo ha score (0.9, 0.8, 0.4) e tipi assegnati **(TP, FP, TP)**; il terzo box corrisponde al secondo GT, distinto dal primo.

Calcolare TP, FP, FN, precision e recall mantenendo score≥0.5, poi score≥0.3. Perché un singolo punto precision/recall non determina AP? Perché nel protocollo reale non si applica il filtro aggiuntivo score≥0.5 prima di COCOeval?

AP usa il ranking delle **detection restituite dal detector** con impostazioni di inferenza fisse: qui score floor 0.001, NMS IoU 0.5 e massimo 100 detection. Non valuta tutte le proposte RPN e non coincide con il solo punto operativo. I due cut del toy non autorizzano a scegliere una nuova soglia sul holdout. Non è richiesto calcolare AP101 né eseguire altre inferenze.

**Risposta P02 (3–5 righe o piccola tabella):** [scrivere qui]


Per N05 richiamare P02 sul ruolo della soglia e aggiungere soltanto la lettura di supporto/fasce già stampati, senza rifare il toy.


**Risposta N05 (3–5 righe o piccola tabella):** [scrivere qui; citare gli output senza ricopiarli]

In [ ]:
# C27
for group in ['small','medium','large']:
    print(group,'supporto',metrics3['support'][group],'AP',metrics3['AP_'+group])
fig,ax=plt.subplots(figsize=(9,4));ax.plot(metrics0['recall_grid'],metrics0['precision_50_recall101'],label='Epoch0');ax.plot(metrics3['recall_grid'],metrics3['precision_50_recall101'],label='Epoch3');ax.set(xlabel='Recall',ylabel='Precision interpolata COCO a IoU=.50',xlim=(0,1),ylim=(0,1.02));ax.legend();ax.grid(alpha=.25);fig.tight_layout();fig.savefig(OUT/'precision_recall.png',dpi=170);plt.show()

In [ ]:
# C28
OPERATING=u.match_at_threshold(holdout,pred3,score=.5,iou=.5)
print({k:v for k,v in OPERATING.items() if k!='per_image'})
assert OPERATING['TP']+OPERATING['FN']==len(np.concatenate([holdout[i][1]['labels'] for i in range(len(holdout))]))

In [ ]:
# C29
LATENCY=u.report_latency(model,holdout,DEVICE)
print('Dispositivo:',DEVICE,'Latenza:',LATENCY)
print('CUDA sincronizzata prima/dopo il timer. Non confrontare dispositivi o confini diversi.')

<!-- C30 -->
## N06 · Errori e soglia operativa

**Lettura di metriche e tempi.** TP/FP/FN sono conteggi dopo il matching uno-a-uno; la soglia score decide il punto operativo. Latenza è il tempo di una richiesta a batch 1; mediana è il 50° percentile e l’intervallo 25°–75° contiene il 50% centrale delle misure. I tempi di riferimento sono CPU locali, non Colab o GPU.

La soglia score.5 serve a questo punto operativo, non alla curva AP. Il conteggio TP/FP/FN richiede matching uno-a-uno per immagine. Seleziona una scena con errori dai conteggi sotto. Una persona reale senza annotazione può apparire come FP: verificare la release e il protocollo, senza cambiare etichette per favorire il risultato.

Approfondire **una sola scena** tra le tre già visualizzate da C31: citare file, soglie, un’osservazione e un limite. Per la distinzione tra punto operativo e AP basta richiamare P02. F04 citerà questa stessa analisi, senza tre commenti aggiuntivi né nuove figure.


**Risposta N06 (3–5 righe o piccola tabella):** [scrivere qui; citare gli output senza ricopiarli]

In [ ]:
# C31
ranked=sorted(OPERATING['per_image'],key=lambda r:(-(r['FP']+r['FN']),r['file']))
print(ranked[:8])
chosen=[next(i for i,f in enumerate(holdout.files) if f.name==r['file']) for r in ranked[:3]]
fig,axes=plt.subplots(1,3,figsize=(15,6))
for ax,i in zip(axes,chosen):
    im,t=holdout[i];ax.imshow(im.permute(1,2,0))
    for x1,y1,x2,y2 in t['boxes'].tolist():ax.add_patch(Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor='#00bdc7',linewidth=2))
    p=pred3[i]
    for box,score in zip(p['boxes'].tolist(),p['scores'].tolist()):
        if score<.5:continue
        x1,y1,x2,y2=box;ax.add_patch(Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor='#ee6749',linewidth=1.5,linestyle='--'));ax.text(x1,y1,f'{score:.2f}',color='white',backgroundcolor='#a53123',fontsize=8)
    ax.set_title(holdout.files[i].name);ax.axis('off')
fig.suptitle('GT turchese continuo · predizioni arancio tratteggiato, score≥.5');fig.tight_layout();fig.savefig(OUT/'error_gallery.png',dpi=180);plt.show()

In [ ]:
# C32
# Stesso checkpoint ufficiale + sola predictor epoch3 ricostruiscono il detector adattato.
restored=u.make_model(BASE,DEVICE)
restored.roi_heads.box_predictor.load_state_dict(torch.load(OUT/'predictor_epoch3.pt',map_location=DEVICE,weights_only=True))
restored.eval()
with torch.no_grad():
    r=restored([holdout[0][0].to(DEVICE)])[0]
max_error=max(float((pred3[0][k]-r[k].cpu()).abs().max()) for k in ['boxes','scores'])
print('Errore massimo ricarica:',max_error)
assert max_error<1e-5
del restored

<!-- C33 -->
## N07 · Due classi, esempio didattico
Questo calcolo è **ipotetico**, non un risultato PennFudan. APpersona=.80, AP bicicletta=.40, entrambe con GT. Calcola mAPmacro. Aggiungi una classe senza GT: perché il suo AP non diventa 0? Perché occorre dichiarare quali categorie partecipano alla media? Lo sfondo è escluso.

**Risposta N07 (3–5 righe o piccola tabella):** [scrivere qui; citare gli output senza ricopiarli]

In [ ]:
# C34
example_ap={'person':.80,'bicycle':.40,'car':None}
valid=[a for a in example_ap.values() if a is not None]
print('mAP sulle 2 classi con supporto:',sum(valid)/len(valid))
assert abs(sum(valid)/len(valid)-.60)<1e-12

<!-- C35 -->
## N08 · Conclusione ed esperimento successivo
Spiega quali evidenze sostengono l’apprendimento della testa di predizione e quali non sostengono superiorità generale del detector. Proponi un solo controllo futuro, per esempio una valutazione multicategoria oppure sui piccoli oggetti, con nuovi dati di acquisizione separata. Se modifichi il modello dopo aver letto Fudan, Fudan diventa validation: serve un holdout nuovo.

È sufficiente proporre un solo controllo pertinente al limite scelto: non occorre eseguirlo né progettare due esperimenti completi. Repliche e challenge ulteriori sono facoltative e non valutate.


**Risposta N08 (3–5 righe o piccola tabella):** [scrivere qui; citare gli output senza ricopiarli]

<!-- C36 -->
## Detection card · Sintesi individuale F01–F05

Scrivere qui la sintesi **entro 500 parole complessive**, . Nessun PDF o report separato: richiamare le tabelle e le figure già presenti. P01/P02 e N01–N08 restano nei loro blocchi e non si svolgono una seconda volta.

**F01 — Dati e provenienza:** [sorgenti/split, annotazioni/diritti, checkpoint e preprocessing, ambiente/dispositivo; citare hash e configurazione]

**F02 — Adattamento:** [sola predictor, budget e inizializzazione, parametri trainabili, invarianti e controlli; citare gli output]

**F03 — Misure:** [AP/AP50/AP75 e per dimensione con supporto/unità, punto operativo separato, costo con confini; richiamare P02 senza rifare i conti]

**F04 — Errori:** [richiamare le tre immagini già prodotte da C31 e approfondire la sola scena scelta in N06; nessuna nuova figura o tre analisi aggiuntive]

**F05 — Conclusione e limite:** [un’affermazione sostenuta e un controllo/esperimento futuro soltanto proposto, senza eseguirlo]

Prima di inviare controllare nome/cognome in C01, C07/C12/C19 completate, risposte P01/P02 e N01–N08, output e sintesi. C37 salva JSON/predizioni e C39 comprime/scarica la cartella degli output; salvare anche il `.ipynb`. Le misure multicategoria del toy sono ipotetiche. Se si usa recupero, dichiarare training e metriche CPU forniti senza attribuirli a GPU o Colab. Non è richiesta una metrica minima né una classifica fra detector.

In [ ]:
# C37
record={'environment':ENV,'protocol':u.CONFIG,'provenance':PROVENANCE,'split_audit':AUDIT,'train_support':train.summary(),'holdout_support':holdout.summary(),'total_parameters':TOTAL,'trainable_parameters':TRAINABLE,'epoch0':{**metrics0,'inference_seconds':time0},'epoch3':{**metrics3,'inference_seconds':time3},'training':TRAINING,'operating_point':OPERATING,'latency':LATENCY,'reload_max_error':max_error}
(OUT/'metriche_finali.json').write_text(json.dumps(record,indent=2))
torch.save({'epoch0':pred0,'epoch3':pred3},OUT/'predictions.pt')
print('Risultati salvati in',OUT)

<!-- C38 -->
## Recupero e ripartenze

**Checkpoint.** Stato salvato per ricostruire il modello. Qui il file della predictor contiene i valori finali della sola testa: richiede la stessa architettura e gli stessi pesi COCO congelati, non va sommato a essi. Per riprendere esattamente un training servirebbe anche lo stato dell’ottimizzatore e della corsa.

Il notebook `recupero/Ripristino_risultati.ipynb` legge i risultati CPU e il checkpoint della testa di predizione: non li presenta come nuova corsa. Per ripetere il training riavvia ed esegui C03, C05, C07, C08, C12, C13, C19, C17, C20, C21, C23. C13 ricrea la testa iniziale: ripetere C23 da sola continuerebbe ad addestrare epoch 3. Per usare un checkpoint salvato serve comunque il checkpoint COCO identico e il codice della stessa architettura. Il file della testa di predizione non è l’intero detector.

Il recupero non sostituisce i completamenti e le risposte individuali. Non ripetere training o inferenze soltanto per P01/P02: sono esercizi scritti. Dichiarare in C36 le parti svolte personalmente e le misure importate; C39 scarica gli output ma non salva automaticamente il notebook.

In [ ]:
# C39
# C39 va eseguita a fine lavoro, soprattutto su Colab.
import shutil
archive=shutil.make_archive(str(Path.cwd()/'Risultati_Lezione16'),'zip',OUT)
print('Archivio output:',archive)
if IN_COLAB:
    files.download(archive)
    print('Salva anche il notebook dal menu File > Scarica > Scarica .ipynb. La VM è temporanea.')

<!-- C40 -->
## Fonti
PennFudan: https://www.cis.upenn.edu/~jshi/ped_html/ e readme.txt della release.

Torchvision detector: https://docs.pytorch.org/vision/main/models/generated/torchvision.models.detection.fasterrcnn_mobilenet_v3_large_320_fpn.html

Tutorial detection: https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html

COCOeval originale: https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/cocoeval.py

Riferimento scientifico Faster R-CNN: https://arxiv.org/abs/1506.01497